# 03. 내 연주를 변주하다 — **확장형 · 음악 차원**

## 이 노트북의 위치

```
기반     : NB01 (듣다)
확장형 ✦ : NB02 (시각 차원) + NB03 (음악 차원) ← 여기
협업형   : NB04 (AI와 대화)
통합     : NB05 (무대)
```

**확장**이란 연주가 단일한 소리 이벤트에 머무르지 않고 **새로운 차원으로 번지는 것**입니다.
NB02에서는 연주를 시각 세계로 확장했고, 여기서는 **작곡적 차원으로 확장**합니다.

## 규칙 기반 음악 변환

작곡 기법 5가지를 MIDI에 적용합니다. 모두 **음악 이론 표준 기법**입니다.

| 변환 | 설명 | 음악적 효과 |
|---|---|---|
| **inversion** | 중심음 기준 상하 반전 | 상행이 하행으로, 멜로디 윤곽 뒤집힘 |
| **retrograde** | 시간 역행 | 끝에서부터 연주 (역행 카논) |
| **transposition** | 지정 음정 이조 | 장단조·조성 변화 |
| **augmentation** | 음가 확대/축소 | 2배 느리게 / 2배 빠르게 |
| **reharmonize** | 코드톤 재배치 | 멜로디 유지, 화성만 재구성 |

**왜 규칙 기반인가?** 음악 이론에 기반한 변환은 LLM의 확률적 생성보다 **예측 가능**하고, **학생이 규칙 자체를 읽고 수정**할 수 있어서 더 교육적입니다. 외부 API도 비용도 없습니다.

**입력**: `artifacts/midi/*.mid` (NB01 출력)  
**출력**: `artifacts/responses/*.mid` (변주 결과)

In [ ]:
!pip install -q pretty_midi midi2audio numpy matplotlib

import warnings
warnings.filterwarnings('ignore')
print('✅ 준비 완료')

In [ ]:
from pathlib import Path
import pretty_midi
import numpy as np

ARTIFACTS = Path('artifacts')
(ARTIFACTS / 'responses').mkdir(parents=True, exist_ok=True)

pieces = {
    'satie': {
        'title': 'Satie — Gymnopédie No.1',
        'midi': ARTIFACTS / 'midi' / 'satie.mid',
    },
    'prokofiev': {
        'title': 'Prokofiev — Toccata Op.11',
        'midi': ARTIFACTS / 'midi' / 'prokofiev.mid',
    },
}

for key, p in pieces.items():
    if not p['midi'].exists():
        raise FileNotFoundError(f"{p['midi']} 없음. NB01을 먼저 실행하세요.")
    print(f"✅ {p['title']}")

---
## 변환 함수 정의

이 셀의 규칙을 읽고, **스스로 변환 함수를 추가**해보세요.

In [ ]:
def extract_notes(midi_path, time_limit=None):
    pm = pretty_midi.PrettyMIDI(str(midi_path))
    notes = []
    for inst in pm.instruments:
        for n in inst.notes:
            if time_limit and n.start >= time_limit:
                continue
            notes.append({'start': n.start, 'end': n.end, 'pitch': n.pitch, 'velocity': n.velocity})
    notes.sort(key=lambda n: n['start'])
    return notes


def notes_to_midi(notes, output_path):
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    for n in notes:
        piano.notes.append(pretty_midi.Note(
            velocity=max(1, min(127, int(n['velocity']))),
            pitch=max(0, min(127, int(n['pitch']))),
            start=max(0, n['start']),
            end=max(n['start'] + 0.05, n['end']),
        ))
    pm.instruments.append(piano)
    pm.write(str(output_path))
    return output_path


# === 5가지 규칙 기반 변환 ===

def invert(notes, center=None):
    """반전: pitch를 center 기준으로 뒤집기."""
    if center is None:
        pitches = [n['pitch'] for n in notes]
        center = int(np.mean(pitches))
    return [{**n, 'pitch': 2 * center - n['pitch']} for n in notes]


def retrograde(notes):
    """역행: 시간축 뒤집기."""
    if not notes: return notes
    duration = max(n['end'] for n in notes)
    result = []
    for n in notes:
        new_start = duration - n['end']
        new_end = duration - n['start']
        result.append({**n, 'start': new_start, 'end': new_end})
    result.sort(key=lambda x: x['start'])
    return result


def transpose(notes, semitones):
    """이조: 모든 pitch에 semitones 더하기."""
    return [{**n, 'pitch': n['pitch'] + semitones} for n in notes]


def time_scale(notes, factor):
    """리듬 확대(factor>1) / 축소(factor<1)."""
    return [{**n, 'start': n['start'] * factor, 'end': n['end'] * factor} for n in notes]


def reharmonize(notes, mode='modal'):
    """코드톤 재화성화: 동시 발음 음을 새 화성 체계로 매핑.
    modal: 장조/단조/도리안 등 모드 순회
    quartal: 4도 적층 (재즈/현대)
    """
    if not notes: return notes
    windows = {}
    for n in notes:
        w = int(n['start'])
        windows.setdefault(w, []).append(n)
    result = []
    mode_shifts = [0, 2, 4, 5, 7, 9, 11] if mode == 'modal' else [0, 5, 10, 15]
    for w, group in windows.items():
        group_sorted = sorted(group, key=lambda n: n['pitch'])
        top = group_sorted[-1]  # 멜로디 음 (최고음) 유지
        result.append(top)
        base = top['pitch'] - 24
        for i, n in enumerate(group_sorted[:-1]):
            shift = mode_shifts[i % len(mode_shifts)]
            result.append({**n, 'pitch': base + shift})
    result.sort(key=lambda n: n['start'])
    return result


# === 변환 조합 레시피 ===
TRANSFORMS = {
    '반전': lambda ns: invert(ns),
    '역행': lambda ns: retrograde(ns),
    '완전5도_상행_이조': lambda ns: transpose(ns, 7),
    '단3도_하행_이조': lambda ns: transpose(ns, -3),
    '두배_느리게': lambda ns: time_scale(ns, 2.0),
    '두배_빠르게': lambda ns: time_scale(ns, 0.5),
    '반전_후_5도이조': lambda ns: transpose(invert(ns), 7),
    '모달_재화성화': lambda ns: reharmonize(ns, 'modal'),
    '쿼털_재화성화': lambda ns: reharmonize(ns, 'quartal'),
}

print('사용 가능한 변환:')
for name in TRANSFORMS:
    print(f'  - {name}')

---
## 변환 적용 — 학생이 선택

In [ ]:
# ★ 여기서 곡과 변환을 선택 ★
target = 'satie'  # 'satie' or 'prokofiev'
transform_name = '반전_후_5도이조'
time_limit = 32.0  # 앞 32초만 변환

original = extract_notes(pieces[target]['midi'], time_limit=time_limit)
transformed = TRANSFORMS[transform_name](original)

out_mid = ARTIFACTS / 'responses' / f'{target}_{transform_name}.mid'
notes_to_midi(transformed, out_mid)

print(f"🎼 {pieces[target]['title']} + {transform_name}")
print(f"   원본: {len(original)}음, 변환 후: {len(transformed)}음")
print(f"   💾 {out_mid}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, (label, notes) in zip(axes, [('원본', original), (f'변환: {transform_name}', transformed)]):
    for n in notes:
        ax.barh(n['pitch'], n['end'] - n['start'], left=n['start'], height=0.8,
                color=plt.cm.viridis(n['velocity']/127), alpha=0.7)
    ax.set_title(label)
    ax.set_ylabel('MIDI pitch')
    ax.grid(alpha=0.2)
axes[-1].set_xlabel('시간 (초)')
plt.tight_layout()
plt.show()

# 소리로 비교
try:
    from midi2audio import FluidSynth
    import IPython.display as ipd
    fs = FluidSynth()
    notes_to_midi(original, '/tmp/orig.mid')
    fs.midi_to_audio('/tmp/orig.mid', '/tmp/orig.wav')
    fs.midi_to_audio(str(out_mid), str(out_mid.with_suffix('.wav')))
    print('🔊 원본:')
    ipd.display(ipd.Audio('/tmp/orig.wav'))
    print(f'🔊 {transform_name}:')
    ipd.display(ipd.Audio(str(out_mid.with_suffix('.wav'))))
except Exception as e:
    print(f'⚠️ FluidSynth 재생 실패: {e}')

---
## 확장 과제

1. **자신만의 변환 함수 추가**: 예를 들어 특정 음정만 건너뛰기, 화성 기능 유지하며 멜로디만 변경
2. **두 곡 조각을 섞는 "콜라주"**: Satie의 화성 구조 위에 Prokofiev의 리듬 얹기
3. **변환 연쇄**: 반전→역행→이조 같은 3단 합성 변환 설계

---

## 다음 단계

**→ NB04 (`04_collaborate.ipynb`)**: 연주와 **대화**합니다.
- AI가 수치로 해석 피드백 (Mode B)
- AI가 시각 매핑 제안 (Mode C)